# 50 -- enumeration probe, step 1: fit / final-read manifests

Zero GPU. Builds the two independent fit pools (`number`, `fo_class`) and the shared
final-read pool, on rung 42 ep4's own checkpoint boundaries.

**Fit pool = 92 `train.parquet` videos (trained on verbatim) + 30 videos promoted into
rung 42's corpus** (scenes seen via generated training questions; these specific
`test.parquet` rows were never trained on). **Final-read = rung 42's own declared 8-video
held-out set** -- the only videos genuinely untouched by training, in any form. Each task
(counting, fo_class) uses its own FULL available pool -- rows are not restricted to frames
carrying both question types; that intersection is reserved for the co-occurrence check in
step 4, not for fitting either probe.

5-fold `StratifiedGroupKFold` per task: grouped by video (no leakage), stratified by a
task-specific difficulty proxy (count bin for counting, gold-set-size for fo_class, matching
Rodrigo's own `[[fo-class-and-number-are-one-front]]` stratification).

In [ ]:
# --- parameters (RAW LITERALS ONLY -- papermill injects a new cell right after THIS one) --
DATA_ROOT = "/workspace/orena-data"
SPLIT_42_JSON = "experiments/42-merged-corpus/RESULTS_split_42.json"
N_SPLITS = 5
SEED = 42

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import sys
from pathlib import Path
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "56-enumeration-probe":
    EXP = REPO / "experiments" / "56-enumeration-probe"

if str(EXP / "_tools") not in sys.path:
    sys.path.insert(0, str(EXP / "_tools"))

import manifest as M
print("repo:", REPO, "| exp:", EXP)

In [ ]:
# --- derived --------------------------------------------------------------------
DATA_ROOT = next(
    (Path(c) for c in (DATA_ROOT, str(REPO / "external_data" / "orena-data"))
     if (Path(c) / "heico" / "data" / "frame" / "test.parquet").exists()
     or (Path(c) / "heico" / "test.parquet").exists()),
    None)
assert DATA_ROOT is not None, "no frame test.parquet found (checked nested and flat layouts)"
SPLIT_42_JSON = REPO / SPLIT_42_JSON

In [ ]:
# --- 1. build the pools -----------------------------------------------------------
pools = M.build_pools(DATA_ROOT, SPLIT_42_JSON)
print(f"fit: {len(pools.fit)} rows, {pools.n_fit_videos} videos")
print(f"final_read: {len(pools.final_read)} rows, {pools.n_final_videos} videos")
print()
print("fit by answer_format:")
print(pools.fit["answer_format"].value_counts())
print()
print("fit by _source:")
print(pools.fit["_source"].value_counts())

In [ ]:
# --- 2. counting task: fold assignment, video-grouped, count-bin-stratified --------
num_fit = pools.fit[pools.fit.answer_format == "number"].copy()
num_fit["_subtmpl"] = num_fit["question"].map(M.tag_number_subtemplate)
num_fit["_bin"] = num_fit["answer"].astype(int).map(M.count_bin)
num_fit["fold"] = M.make_folds(num_fit, strat_col="_bin", n_splits=N_SPLITS, seed=SEED)

leak = num_fit.groupby("_video_key")["fold"].nunique()
assert (leak <= 1).all(), f"{(leak > 1).sum()} videos leak across counting-task folds"
print(f"counting fit pool: {len(num_fit)} rows, {num_fit._video_key.nunique()} videos, "
      f"{N_SPLITS} folds, 0 leaked")
print(num_fit.groupby("fold")["_video_key"].nunique(), "videos/fold")
print()
print("sub-template counts:")
print(num_fit["_subtmpl"].value_counts())

In [ ]:
# --- 3. fo_class task: fold assignment, video-grouped, n-classes-stratified ---------
fo_fit = pools.fit[pools.fit.answer_format == "fo_class"].copy()
fo_fit["_nclasses"] = fo_fit["answer"].map(M.n_classes_in_gold)
fo_fit["fold"] = M.make_folds(fo_fit, strat_col="_nclasses", n_splits=N_SPLITS, seed=SEED)

leak2 = fo_fit.groupby("_video_key")["fold"].nunique()
assert (leak2 <= 1).all(), f"{(leak2 > 1).sum()} videos leak across fo_class-task folds"
print(f"fo_class fit pool: {len(fo_fit)} rows, {fo_fit._video_key.nunique()} videos, "
      f"{N_SPLITS} folds, 0 leaked")
print(fo_fit.groupby("fold")["_video_key"].nunique(), "videos/fold")
print()
print("gold-set-size distribution:")
print(fo_fit["_nclasses"].value_counts().sort_index())

In [ ]:
# --- 4. final-read pool: tag sub-templates, no folds (touched once, in step 4) ------
final = pools.final_read.copy()
final.loc[final.answer_format == "number", "_subtmpl"] = (
    final.loc[final.answer_format == "number", "question"].map(M.tag_number_subtemplate)
)
final.loc[final.answer_format == "fo_class", "_nclasses"] = (
    final.loc[final.answer_format == "fo_class", "answer"].map(M.n_classes_in_gold)
)
n_number = int((final.answer_format == "number").sum())
n_foclass = int((final.answer_format == "fo_class").sum())
n_cooccur = final.groupby("_video_key").apply(
    lambda g: (g.answer_format == "number").any() and (g.answer_format == "fo_class").any(),
    include_groups=False,
).sum()
print(f"final-read: {n_number} number rows, {n_foclass} fo_class rows, "
      f"{final._video_key.nunique()} videos")

In [ ]:
# --- 5. persist: manifests + sha256 sidecars ---------------------------------------
# `timestamp_start` is the frame-identity key `04_final_read` needs for the aggregation
# co-occurrence check (same video + same timestamp = same physical frame, per the
# organizers' own annotation pipeline) -- everything else here is identity/strat columns.
cols = ["qID", "dataset", "video", "_video_key", "_source", "timestamp_start", "question",
        "answer", "answer_format", "_subtmpl", "_bin", "_nclasses", "fold"]

num_cols = [c for c in cols if c in num_fit.columns]
fo_cols = [c for c in cols if c in fo_fit.columns]
final_cols = [c for c in cols if c in final.columns]

p1 = M.freeze(num_fit, EXP / "RESULTS_fit_number_v1.csv", num_cols)
p2 = M.freeze(fo_fit, EXP / "RESULTS_fit_foclass_v1.csv", fo_cols)
p3 = M.freeze(final, EXP / "RESULTS_final_read_v1.csv", final_cols)
print("wrote", p1)
print("wrote", p2)
print("wrote", p3)